# 02 - Feature Engineering

Goal: convert the Phase 1 LendingClub sample into a clean, model-ready baseline dataset for `03_model_training.ipynb`.

This notebook stays focused on credit risk modeling only. It does not include SHAP, document AI, fraud rules, LLM reports, or Streamlit.

## 1. Setup

Use a bounded sample and chunked reads so the notebook works on an 8 GB RAM machine.

In [1]:
import json
import os
import warnings
from pathlib import Path

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
DATA_PATH = Path("../data/lendingclub/raw/accepted_2007_to_2018Q4.csv")
FEATURE_OUTPUT_PATH = Path("../data/processed/features/lendingclub_baseline_features.parquet")
METADATA_OUTPUT_PATH = Path("../data/processed/features/lendingclub_baseline_metadata.json")

N_ROWS = 500_000
CHUNK_SIZE = 50_000
RANDOM_STATE = 42
HIGH_MISSING_THRESHOLD = 50.0
MAX_CATEGORICAL_CARDINALITY = 40

FEATURE_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

DATA_PATH.exists(), DATA_PATH

(True, WindowsPath('../data/lendingclub/raw/accepted_2007_to_2018Q4.csv'))

## 2. First Pass: Profile Final-Outcome Rows

Read the raw CSV in chunks, keep only known final outcomes for profiling, and compute missingness without holding the full raw sample in memory.

In [3]:
final_statuses = ["Fully Paid", "Charged Off"]

raw_rows_loaded = 0
raw_column_count = None
status_counts = pd.Series(dtype="int64")
missing_counts = None
filtered_rows = 0
all_columns = None

reader = pd.read_csv(
    DATA_PATH,
    nrows=N_ROWS,
    chunksize=CHUNK_SIZE,
    low_memory=True,
)

for chunk in reader:
    raw_rows_loaded += len(chunk)
    raw_column_count = chunk.shape[1]
    all_columns = chunk.columns.tolist()

    status_counts = status_counts.add(
        chunk["loan_status"].value_counts(dropna=False),
        fill_value=0,
    )

    final_chunk = chunk[chunk["loan_status"].isin(final_statuses)]
    filtered_rows += len(final_chunk)

    chunk_missing = final_chunk.isna().sum()
    if missing_counts is None:
        missing_counts = chunk_missing
    else:
        missing_counts = missing_counts.add(chunk_missing, fill_value=0)

    del final_chunk
    del chunk

raw_sample_shape = (raw_rows_loaded, raw_column_count)

print(f"Raw sample shape: {raw_sample_shape}")
print(f"Final-outcome rows: {filtered_rows:,}")
status_counts.sort_values(ascending=False)

Raw sample shape: (500000, 151)
Final-outcome rows: 391,164


loan_status
Fully Paid           312340.0000
Current              104240.0000
Charged Off           78824.0000
Late (31-120 days)     2977.0000
In Grace Period        1046.0000
Late (16-30 days)       567.0000
Default                   4.0000
NaN                       2.0000
dtype: float64

In [4]:
missing_summary = (
    (missing_counts / filtered_rows * 100)
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ["column", "missing_percentage"]

high_missing_cols = missing_summary.loc[
    missing_summary["missing_percentage"] >= HIGH_MISSING_THRESHOLD,
    "column",
].tolist()

print(f"Columns with >= {HIGH_MISSING_THRESHOLD:.0f}% missing values: {len(high_missing_cols)}")
missing_summary.head(40)

Columns with >= 50% missing values: 57


,column,missing_percentage
0,member_id,100.0000
1,next_pymnt_d,100.0000
2,desc,99.9893
3,sec_app_mths_since_last_major_derog,99.8009
4,orig_projected_additional_accrued_interest,99.6365
5,hardship_last_payment_amount,99.5204
6,hardship_type,99.5204
7,hardship_status,99.5204
8,hardship_amount,99.5204
9,hardship_start_date,99.5204


## 3. Identify Leakage and Base Exclusions

The baseline model should use information available around loan application/origination time. Post-origination repayment, recovery, hardship, settlement, and later payment fields are excluded.

In [5]:
leakage_keywords = [
    "out_prncp",
    "total_pymnt",
    "recover",
    "collection",
    "last_pymnt",
    "next_pymnt",
    "last_credit_pull",
    "settlement",
    "hardship",
    "debt_settlement",
]

leakage_candidates = sorted([
    col for col in all_columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
])
confirmed_leakage_cols = leakage_candidates.copy()

id_or_text_cols = [
    "id",
    "member_id",
    "url",
    "desc",
    "title",
    "emp_title",
    "zip_code",
]
id_or_text_cols = [col for col in id_or_text_cols if col in all_columns]

base_exclude_cols = sorted(set(
    ["loan_status"] + confirmed_leakage_cols + id_or_text_cols
))
all_exclude_cols = sorted(set(base_exclude_cols + high_missing_cols))

print(f"Confirmed leakage columns: {len(confirmed_leakage_cols)}")
print(f"ID/text columns: {len(id_or_text_cols)}")
print(f"Total excluded columns: {len(all_exclude_cols)}")
all_exclude_cols

Confirmed leakage columns: 31
ID/text columns: 7
Total excluded columns: 75


['all_util',
 'annual_inc_joint',
 'collection_recovery_fee',
 'collections_12_mths_ex_med',
 'debt_settlement_flag',
 'debt_settlement_flag_date',
 'deferral_term',
 'desc',
 'dti_joint',
 'emp_title',
 'hardship_amount',
 'hardship_dpd',
 'hardship_end_date',
 'hardship_flag',
 'hardship_last_payment_amount',
 'hardship_length',
 'hardship_loan_status',
 'hardship_payoff_balance_amount',
 'hardship_reason',
 'hardship_start_date',
 'hardship_status',
 'hardship_type',
 'id',
 'il_util',
 'inq_fi',
 'inq_last_12m',
 'last_credit_pull_d',
 'last_pymnt_amnt',
 'last_pymnt_d',
 'loan_status',
 'max_bal_bc',
 'member_id',
 'mths_since_last_major_derog',
 'mths_since_last_record',
 'mths_since_rcnt_il',
 'mths_since_recent_bc_dlq',
 'mths_since_recent_revol_delinq',
 'next_pymnt_d',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'open_rv_12m',
 'open_rv_24m',
 'orig_projected_additional_accrued_interest',
 'out_prncp',
 'out_prncp_inv',
 'payment_plan_start_date',
 'recov

## 4. Second Pass: Load Selected Features Only

Now load only the columns that survived the exclusion rules. This keeps the working dataframe much smaller than the raw LendingClub file.

In [6]:
feature_cols = [
    col for col in all_columns
    if col not in all_exclude_cols and col != "target"
]

usecols = feature_cols + ["loan_status"]

filtered_chunks = []
reader = pd.read_csv(
    DATA_PATH,
    nrows=N_ROWS,
    chunksize=CHUNK_SIZE,
    usecols=usecols,
    low_memory=True,
)

for chunk in reader:
    chunk = chunk[chunk["loan_status"].isin(final_statuses)].copy()
    if not chunk.empty:
        filtered_chunks.append(chunk)
    del chunk

df_clean = pd.concat(filtered_chunks, ignore_index=True)
df_clean["target"] = df_clean["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1,
}).astype("int8")

X_raw = df_clean[feature_cols].copy()
y = df_clean["target"].copy()

print(f"Selected raw feature columns: {len(feature_cols)}")
print(f"X_raw shape: {X_raw.shape}")
print(f"y shape: {y.shape}")
y.value_counts(normalize=True).mul(100)

Selected raw feature columns: 76
X_raw shape: (391164, 76)
y shape: (391164,)


target
0   79.8489
1   20.1511
Name: proportion, dtype: float64

## 5. Convert Dates into Numeric Features

Raw month-year strings are converted into useful numeric fields. `credit_history_months` captures credit history length at loan issue time.

In [7]:
date_like_cols = ["issue_d", "earliest_cr_line"]
date_like_cols = [col for col in date_like_cols if col in X_raw.columns]

for col in date_like_cols:
    X_raw[col] = pd.to_datetime(X_raw[col], format="%b-%Y", errors="coerce")

if "issue_d" in X_raw.columns:
    X_raw["issue_year"] = X_raw["issue_d"].dt.year.astype("float32")
    X_raw["issue_month"] = X_raw["issue_d"].dt.month.astype("float32")

if "earliest_cr_line" in X_raw.columns and "issue_d" in X_raw.columns:
    X_raw["credit_history_months"] = (
        (X_raw["issue_d"].dt.year - X_raw["earliest_cr_line"].dt.year) * 12
        + (X_raw["issue_d"].dt.month - X_raw["earliest_cr_line"].dt.month)
    ).astype("float32")

X_raw = X_raw.drop(columns=date_like_cols)

created_date_features = [
    col for col in ["issue_year", "issue_month", "credit_history_months"]
    if col in X_raw.columns
]
X_raw[created_date_features].head()

,issue_year,issue_month,credit_history_months
0,2015.0000,12.0000,148.0000
1,2015.0000,12.0000,192.0000
2,2015.0000,12.0000,184.0000
3,2015.0000,12.0000,210.0000
4,2015.0000,12.0000,338.0000


## 6. Convert Structured Text Fields

Some fields are numeric concepts stored as text. Convert them before generic categorical encoding.

In [8]:
percentage_cols = ["int_rate", "revol_util"]
percentage_cols = [col for col in percentage_cols if col in X_raw.columns]

for col in percentage_cols:
    if X_raw[col].dtype == "object":
        X_raw[col] = (
            X_raw[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .replace("nan", pd.NA)
            .astype("float32")
        )

if "term" in X_raw.columns:
    X_raw["term_months"] = (
        X_raw["term"]
        .astype(str)
        .str.extract(r"(\d+)")[0]
        .astype("float32")
    )

if "emp_length" in X_raw.columns:
    emp_length_map = {
        "< 1 year": 0,
        "1 year": 1,
        "2 years": 2,
        "3 years": 3,
        "4 years": 4,
        "5 years": 5,
        "6 years": 6,
        "7 years": 7,
        "8 years": 8,
        "9 years": 9,
        "10+ years": 10,
    }
    X_raw["emp_length_years"] = X_raw["emp_length"].map(emp_length_map).astype("float32")

X_raw = X_raw.drop(columns=[col for col in ["term", "emp_length"] if col in X_raw.columns])

preview_cols = [
    col for col in ["int_rate", "revol_util", "term_months", "emp_length_years"]
    if col in X_raw.columns
]
X_raw[preview_cols].head()

,int_rate,revol_util,term_months,emp_length_years
0,13.9900,29.7000,36.0000,10.0000
1,11.9900,19.2000,36.0000,10.0000
2,10.7800,56.2000,60.0000,10.0000
3,22.4500,64.5000,60.0000,3.0000
4,13.4400,68.4000,36.0000,4.0000


## 7. Drop High-Cardinality Categories

Low-cardinality categorical fields are safe for a baseline one-hot encoding. High-cardinality text-like fields are dropped to avoid memory blowup.

In [9]:
numeric_features = X_raw.select_dtypes(include="number").columns.tolist()
categorical_features = X_raw.select_dtypes(include="object").columns.tolist()

categorical_cardinality = (
    X_raw[categorical_features]
    .nunique(dropna=True)
    .sort_values(ascending=False)
    .reset_index()
)
categorical_cardinality.columns = ["column", "unique_values"]

high_cardinality_categorical_cols = categorical_cardinality.loc[
    categorical_cardinality["unique_values"] > MAX_CATEGORICAL_CARDINALITY,
    "column",
].tolist()

X_work = X_raw.drop(columns=high_cardinality_categorical_cols).copy()

numeric_features = X_work.select_dtypes(include="number").columns.tolist()
categorical_features = X_work.select_dtypes(include="object").columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features kept: {len(categorical_features)}")
print(f"High-cardinality categorical columns dropped: {len(high_cardinality_categorical_cols)}")
categorical_cardinality

Numeric features: 67
Categorical features kept: 9
High-cardinality categorical columns dropped: 1


,column,unique_values
0,addr_state,50
1,sub_grade,35
2,purpose,14
3,grade,7
4,home_ownership,4
5,verification_status,3
6,application_type,2
7,initial_list_status,2
8,disbursement_method,2
9,pymnt_plan,1


## 8. Impute Missing Values

Use simple baseline imputation: median for numeric features and `Unknown` for categorical features.

In [10]:
numeric_impute_values = X_work[numeric_features].median(numeric_only=True)
X_work[numeric_features] = X_work[numeric_features].fillna(numeric_impute_values)

for col in categorical_features:
    X_work[col] = X_work[col].fillna("Unknown").astype(str)

missing_after_imputation = int(X_work.isna().sum().sum())
print(f"Missing values after imputation: {missing_after_imputation}")

Missing values after imputation: 0


## 9. One-Hot Encode Categorical Features

Use `drop_first=True` for a compact baseline feature matrix.

In [11]:
X_encoded = pd.get_dummies(
    X_work,
    columns=categorical_features,
    drop_first=True,
    dtype="uint8",
)

print(f"Encoded feature matrix shape: {X_encoded.shape}")
print(f"Object columns after encoding: {X_encoded.select_dtypes(include='object').shape[1]}")
X_encoded.head()

Encoded feature matrix shape: (391164, 128)
Object columns after encoding: 0


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,open_acc,pub_rec,revol_bal,revol_util,total_acc,total_rec_prncp,total_rec_int,total_rec_late_fee,last_fico_range_high,last_fico_range_low,policy_code,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_inq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,...,grade_C,grade_D,grade_E,grade_F,grade_G,sub_grade_A2,sub_grade_A3,sub_grade_A4,sub_grade_A5,sub_grade_B1,sub_grade_B2,sub_grade_B3,sub_grade_B4,sub_grade_B5,sub_grade_C1,sub_grade_C2,sub_grade_C3,sub_grade_C4,sub_grade_C5,sub_grade_D1,sub_grade_D2,sub_grade_D3,sub_grade_D4,sub_grade_D5,sub_grade_E1,sub_grade_E2,sub_grade_E3,sub_grade_E4,sub_grade_E5,sub_grade_F1,sub_grade_F2,sub_grade_F3,sub_grade_F4,sub_grade_F5,sub_grade_G1,sub_grade_G2,sub_grade_G3,sub_grade_G4,sub_grade_G5,home_ownership_MORTGAGE,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w,application_type_Joint App,disbursement_method_DirectPay
0,3600.0000,3600.0000,3600.0000,13.9900,123.0300,55000.0000,5.9100,0.0000,675.0000,679.0000,1.0000,30.0000,7.0000,0.0000,2765.0000,29.7000,13.0000,3600.0000,821.7200,0.0000,564.0000,560.0000,1.0000,0.0000,722.0000,144904.0000,9300.0000,4.0000,20701.0000,1506.0000,37.2000,0.0000,0.0000,148.0000,128.0000,3.0000,3.0000,1.0000,4.0000,4.0000,2.0000,2.0000,4.0000,2.0000,5.0000,3.0000,4.0000,9.0000,4.0000,7.0000,0.0000,0.0000,0.0000,3.0000,76.9000,0.0000,0.0000,0.0000,178050.0000,7746.0000,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,24700.0000,24700.0000,24700.0000,11.9900,820.2800,65000.0000,16.0600,1.0000,715.0000,719.0000,4.0000,6.0000,22.0000,0.0000,21470.0000,19.2000,38.0000,24700.0000,979.6600,0.0000,699.0000,695.0000,1.0000,0.0000,0.0000,204396.0000,111800.0000,4.0000,9733.0000,57830.0000,27.1000,0.0000,0.0000,113.0000,192.0000,2.0000,2.0000,4.0000,2.0000,0.0000,0.0000,5.0000,5.0000,13.0000,17.0000,6.0000,20.0000,27.0000,5.0000,22.0000,0.0000,0.0000,0.0000,2.0000,97.4000,7.7000,0.0000,0.0000,314017.0000,39475.0000,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0
2,20000.0000,20000.0000,20000.0000,10.7800,432.6600,63000.0000,10.7800,0.0000,695.0000,699.0000,0.0000,31.0000,6.0000,0.0000,7869.0000,56.2000,18.0000,20000.0000,2705.9200,0.0000,704.0000,700.0000,1.0000,0.0000,0.0000,189699.0000,14000.0000,6.0000,31617.0000,2737.0000,55.9000,0.0000,0.0000,125.0000,184.0000,14.0000,14.0000,5.0000,101.0000,10.0000,0.0000,2.0000,3.0000,2.0000,4.0000,6.0000,4.0000,7.0000,3.0000,6.0000,0.0000,0.0000,0.0000,0.0000,100.0000,50.0000,0.0000,0.0000,218418.0000,18696.0000,...,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0
3,10400.0000,10400.0000,10400.0000,22.4500,289.9100,104433.0000,25.3700,1.0000,695.0000,699.0000,3.0000,12.0000,12.0000,0.0000,21929.0000,64.5000,35.0000,10400.0000,1340.5000,0.0000,704.0000,700.0000,1.0000,0.0000,0.0000,331730.0000,34000.0000,10.0000,27644.0000,4567.0000,77.5000,0.0000,0.0000,128.

## 10. Downcast Numeric Values

Downcasting reduces memory usage before saving the final model-ready table.

In [12]:
memory_before_mb = X_encoded.memory_usage(deep=True).sum() / 1024**2

float_cols = X_encoded.select_dtypes(include=["float64"]).columns
int_cols = X_encoded.select_dtypes(include=["int64", "int32"]).columns

for col in float_cols:
    X_encoded[col] = pd.to_numeric(X_encoded[col], downcast="float")

for col in int_cols:
    X_encoded[col] = pd.to_numeric(X_encoded[col], downcast="integer")

memory_after_mb = X_encoded.memory_usage(deep=True).sum() / 1024**2

print(f"Memory before downcast: {memory_before_mb:.2f} MB")
print(f"Memory after downcast: {memory_after_mb:.2f} MB")
print(f"Memory reduction: {memory_before_mb - memory_after_mb:.2f} MB")

Memory before downcast: 215.25 MB
Memory after downcast: 128.70 MB
Memory reduction: 86.55 MB


## 11. Build Final Model Dataset

`model_df` is the final output for the next Phase 1 notebook: model training.

In [13]:
model_df = X_encoded.copy()
model_df["target"] = y.astype("int8").values

print(f"Final model_df shape: {model_df.shape}")
model_df["target"].value_counts(normalize=True).mul(100)

Final model_df shape: (391164, 129)


target
0   79.8489
1   20.1511
Name: proportion, dtype: float64

## 12. Validate Feature Table

Run basic checks before saving the dataset.

In [14]:
object_cols_remaining = model_df.select_dtypes(include="object").columns.tolist()
missing_values_remaining = int(model_df.isna().sum().sum())
leakage_cols_remaining = sorted(set(model_df.columns).intersection(set(confirmed_leakage_cols)))
base_excluded_remaining = sorted(set(model_df.columns).intersection(set(base_exclude_cols)))

validation_summary = {
    "rows": int(model_df.shape[0]),
    "columns": int(model_df.shape[1]),
    "feature_columns": int(model_df.shape[1] - 1),
    "target_column_present": "target" in model_df.columns,
    "object_columns_remaining": len(object_cols_remaining),
    "missing_values_remaining": missing_values_remaining,
    "leakage_columns_remaining": len(leakage_cols_remaining),
    "base_excluded_columns_remaining": len(base_excluded_remaining),
}

validation_summary

{'rows': 391164,
 'columns': 129,
 'feature_columns': 128,
 'target_column_present': True,
 'object_columns_remaining': 0,
 'missing_values_remaining': 0,
 'leakage_columns_remaining': 0,
 'base_excluded_columns_remaining': 0}

In [15]:
assert "target" in model_df.columns
assert len(object_cols_remaining) == 0
assert missing_values_remaining == 0
assert len(leakage_cols_remaining) == 0
assert len(base_excluded_remaining) == 0
assert model_df.shape[0] == y.shape[0]

print("Validation passed.")

Validation passed.


## 13. Save Processed Dataset and Metadata

Save one combined parquet file containing encoded features plus the `target` column, along with metadata for reproducibility.

In [16]:
model_df.to_parquet(FEATURE_OUTPUT_PATH, index=False)

metadata = {
    "source_path": str(DATA_PATH),
    "n_rows_loaded": N_ROWS,
    "chunk_size": CHUNK_SIZE,
    "random_state": RANDOM_STATE,
    "high_missing_threshold": HIGH_MISSING_THRESHOLD,
    "max_categorical_cardinality": MAX_CATEGORICAL_CARDINALITY,
    "raw_sample_shape": list(raw_sample_shape),
    "filtered_final_outcome_rows": int(filtered_rows),
    "selected_modeling_shape": list(df_clean.shape),
    "final_model_shape": list(model_df.shape),
    "target_mapping": {"Fully Paid": 0, "Charged Off": 1},
    "target_distribution_percentage": (
        model_df["target"].value_counts(normalize=True).sort_index().mul(100).to_dict()
    ),
    "confirmed_leakage_cols": confirmed_leakage_cols,
    "id_or_text_cols": id_or_text_cols,
    "high_missing_cols": high_missing_cols,
    "high_cardinality_categorical_cols": high_cardinality_categorical_cols,
    "numeric_impute_values": numeric_impute_values.astype(float).to_dict(),
    "feature_columns": [col for col in model_df.columns if col != "target"],
}

with open(METADATA_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved features to: {FEATURE_OUTPUT_PATH}")
print(f"Saved metadata to: {METADATA_OUTPUT_PATH}")
print(f"Feature file exists: {FEATURE_OUTPUT_PATH.exists()}")
print(f"Metadata file exists: {METADATA_OUTPUT_PATH.exists()}")

Saved features to: ..\data\processed\features\lendingclub_baseline_features.parquet
Saved metadata to: ..\data\processed\features\lendingclub_baseline_metadata.json
Feature file exists: True
Metadata file exists: True


## 14. Final Handoff

This notebook produces `model_df`, a fully numeric, imputed, encoded, leakage-controlled baseline dataset for `03_model_training.ipynb`.

Next notebook: train baseline credit risk models using `data/processed/features/lendingclub_baseline_features.parquet`.

In [18]:
df_clean.groupby(
'loan_status'
)[
[
'total_rec_prncp',
'total_rec_int',
'last_fico_range_high',
'last_fico_range_low',
'total_rec_late_fee'
]
].mean()

,total_rec_prncp,total_rec_int,last_fico_range_high,last_fico_range_low,total_rec_late_fee
loan_status,,,,,
Charged Off,4794.9373,2805.5599,565.8701,503.1230,4.5463
Fully Paid,14383.9249,2300.8276,703.3988,697.4179,0.9560
